In [0]:
import mlflow
import mlflow.spark
import pandas as pd

from pyspark.sql.functions import col
from pyspark.ml.evaluation import BinaryClassificationEvaluator
from pyspark.ml.functions import vector_to_array

df_test = spark.table(
    "proyecto_final_mlops.telco_database.churn_dataset_test_data"
)

print("Registros Test:", df_test.count())

display(df_test.limit(5))

Registros Test: 64374


CustomerID,Age,Gender,Tenure,Usage_Frequency,Support_Calls,Payment_Delay,Subscription_Type,Contract_Length,Total_Spend,Last_Interaction,Churn
1,22,Female,25,14,4,27,Basic,Monthly,598,9,1
2,41,Female,28,28,7,13,Standard,Monthly,584,20,0
3,47,Male,27,10,2,29,Premium,Annual,757,21,0
4,35,Male,9,12,5,17,Premium,Quarterly,232,18,0
5,53,Female,58,24,9,2,Standard,Annual,533,18,0


In [0]:
import os

os.environ["MLFLOW_DFS_TMP"] = "/Volumes/proyecto_final_mlops/telco_database/tmp"

mlflow.set_registry_uri("databricks-uc")

model_name = "proyecto_final_mlops.telco_database.churn_logit_model"
model_version = 1

model_uri = f"models:/{model_name}/{model_version}"

model = mlflow.spark.load_model(model_uri)

print("Modelo cargado:", model_uri)

Modelo cargado: models:/proyecto_final_mlops.telco_database.churn_logit_model/1


In [0]:
predictions_test = model.transform(df_test)

predictions_test = predictions_test.withColumn(
    "prob_churn",
    vector_to_array("probability")[1]
)

display(
    predictions_test.select(
        "CustomerID",
        "Churn",
        "prediction",
        "prob_churn"
    ).limit(20)
)

CustomerID,Churn,prediction,prob_churn
1,1,1.0,0.9999999999993011
2,0,1.0,0.9999999999998793
3,0,1.0,0.669917291583708
4,0,1.0,0.9861918739804226
5,0,1.0,0.9933323039336314
6,0,1.0,0.9999999999999734
7,1,1.0,0.9995035386414889
8,0,1.0,0.999999999998225
9,0,1.0,0.9999999999998016
10,0,0.0,0.252718597523374


In [0]:
roc_auc = BinaryClassificationEvaluator(
    labelCol="Churn",
    rawPredictionCol="rawPrediction",
    metricName="areaUnderROC"
).evaluate(predictions_test)

pr_auc = BinaryClassificationEvaluator(
    labelCol="Churn",
    rawPredictionCol="rawPrediction",
    metricName="areaUnderPR"
).evaluate(predictions_test)

tp = predictions_test.filter(
    (col("Churn") == 1) & (col("prediction") == 1)
).count()

tn = predictions_test.filter(
    (col("Churn") == 0) & (col("prediction") == 0)
).count()

fp = predictions_test.filter(
    (col("Churn") == 0) & (col("prediction") == 1)
).count()

fn = predictions_test.filter(
    (col("Churn") == 1) & (col("prediction") == 0)
).count()

accuracy = (tp + tn) / (tp + tn + fp + fn)
precision = tp / (tp + fp) if (tp + fp) > 0 else 0
recall = tp / (tp + fn) if (tp + fn) > 0 else 0

f1 = (
    2 * precision * recall / (precision + recall)
    if (precision + recall) > 0 else 0
)

In [0]:
print("RESULTADOS TEST FINAL")
print("----------------------")
print(f"ROC AUC   : {roc_auc:.4f}")
print(f"PR AUC    : {pr_auc:.4f}")
print(f"Accuracy  : {accuracy:.4f}")
print(f"Precision : {precision:.4f}")
print(f"Recall    : {recall:.4f}")
print(f"F1        : {f1:.4f}")

RESULTADOS TEST FINAL
----------------------
ROC AUC   : 0.6831
PR AUC    : 0.6345
Accuracy  : 0.5712
Precision : 0.5251
Recall    : 0.9906
F1        : 0.6864


In [0]:
confusion_matrix = (
    predictions_test
    .groupBy("Churn", "prediction")
    .count()
    .orderBy("Churn", "prediction")
)

display(confusion_matrix)

Churn,prediction,count
0,0.0,6566
0,1.0,27315
1,0.0,287
1,1.0,30206


In [0]:
import mlflow.sklearn

model_name_rf = (
    "proyecto_final_mlops."
    "telco_database."
    "churn_random_forest_model"
)

model_version_rf = 1

model_uri_rf = f"models:/{model_name_rf}/{model_version_rf}"

rf_model = mlflow.sklearn.load_model(model_uri_rf)

print("Modelo RF cargado:", model_uri_rf)

Modelo RF cargado: models:/proyecto_final_mlops.telco_database.churn_random_forest_model/1


In [0]:
categorical_cols = [
    "Gender",
    "Subscription_Type",
    "Contract_Length"
]

numeric_cols = [
    "Age",
    "Tenure",
    "Usage_Frequency",
    "Support_Calls",
    "Payment_Delay",
    "Total_Spend",
    "Last_Interaction"
]

feature_cols = numeric_cols + categorical_cols

test_pd = df_test.select(
    ["CustomerID"] + feature_cols + ["Churn"]
).toPandas()

X_test_rf = test_pd[feature_cols].copy()
y_test_rf = test_pd["Churn"]

X_test_rf = pd.get_dummies(
    X_test_rf,
    columns=categorical_cols
)

# Mismas columnas usadas al entrenar
X_test_rf = X_test_rf.reindex(
    columns=rf_model.feature_names_in_,
    fill_value=0
)

In [0]:
pred_rf = rf_model.predict(X_test_rf)
prob_rf = rf_model.predict_proba(X_test_rf)[:, 1]

In [0]:
from sklearn.metrics import (
    roc_auc_score,
    average_precision_score,
    accuracy_score,
    precision_score,
    recall_score,
    f1_score
)

roc_auc_rf = roc_auc_score(y_test_rf, prob_rf)
pr_auc_rf = average_precision_score(y_test_rf, prob_rf)
accuracy_rf = accuracy_score(y_test_rf, pred_rf)
precision_rf = precision_score(y_test_rf, pred_rf)
recall_rf = recall_score(y_test_rf, pred_rf)
f1_rf = f1_score(y_test_rf, pred_rf)

print("RESULTADOS RANDOM FOREST - TEST FINAL")
print("--------------------------------------")
print(f"ROC AUC   : {roc_auc_rf:.4f}")
print(f"PR AUC    : {pr_auc_rf:.4f}")
print(f"Accuracy  : {accuracy_rf:.4f}")
print(f"Precision : {precision_rf:.4f}")
print(f"Recall    : {recall_rf:.4f}")
print(f"F1        : {f1_rf:.4f}")

RESULTADOS RANDOM FOREST - TEST FINAL
--------------------------------------
ROC AUC   : 0.6093
PR AUC    : 0.5363
Accuracy  : 0.5040
Precision : 0.4885
Recall    : 0.9987
F1        : 0.6561


In [0]:
# Recalcular métricas Logit desde predictions_test
from pyspark.ml.evaluation import BinaryClassificationEvaluator

roc_auc_logit = BinaryClassificationEvaluator(
    labelCol="Churn",
    rawPredictionCol="rawPrediction",
    metricName="areaUnderROC"
).evaluate(predictions_test)

pr_auc_logit = BinaryClassificationEvaluator(
    labelCol="Churn",
    rawPredictionCol="rawPrediction",
    metricName="areaUnderPR"
).evaluate(predictions_test)

tp = predictions_test.filter(
    (col("Churn") == 1) & (col("prediction") == 1)
).count()

tn = predictions_test.filter(
    (col("Churn") == 0) & (col("prediction") == 0)
).count()

fp = predictions_test.filter(
    (col("Churn") == 0) & (col("prediction") == 1)
).count()

fn = predictions_test.filter(
    (col("Churn") == 1) & (col("prediction") == 0)
).count()

accuracy_logit = (tp + tn) / (tp + tn + fp + fn)
precision_logit = tp / (tp + fp)
recall_logit = tp / (tp + fn)
f1_logit = 2 * precision_logit * recall_logit / (precision_logit + recall_logit)


comparacion = spark.createDataFrame(
    [
        (
            "Logit",
            float(roc_auc_logit),
            float(pr_auc_logit),
            float(accuracy_logit),
            float(precision_logit),
            float(recall_logit),
            float(f1_logit)
        ),
        (
            "Random Forest",
            float(roc_auc_rf),
            float(pr_auc_rf),
            float(accuracy_rf),
            float(precision_rf),
            float(recall_rf),
            float(f1_rf)
        )
    ],
    [
        "Modelo",
        "ROC_AUC",
        "PR_AUC",
        "Accuracy",
        "Precision",
        "Recall",
        "F1"
    ]
)

display(comparacion)

Modelo,ROC_AUC,PR_AUC,Accuracy,Precision,Recall,F1
Logit,0.6830549281096526,0.6345477793827655,0.5712244073694349,0.5251299525390727,0.9905880038041518,0.6863908014634036
Random Forest,0.6092988759467313,0.5363171299990934,0.5039922950259421,0.4884750493239016,0.998688223526711,0.6560601490800982
